## IMBD_2024_Reccomendation System by Storyline

## STEP 1: Create dataset using Selenium

In [1]:
pip install selenium webdriver-manager pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.action_chains import ActionChains
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException, ElementClickInterceptedException
import pandas as pd

# --- Re-initialize WebDriver and navigate to URL ---
# Set up Chrome options for headless mode and add a User-Agent
chrome_options = Options()
chrome_options.add_argument('--headless')
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')
chrome_options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/103.0.0.0 Safari/537.36') # Adding a user-agent

# Initialize the Chrome WebDriver using ChromeDriverManager
driver = webdriver.Chrome(service=webdriver.ChromeService(ChromeDriverManager().install()), options=chrome_options)

# Define the IMDb URL for feature films released in 2024
url = "https://www.imdb.com/search/title/?title_type=feature&release_date=2024-01-01,2024-12-31"

# Open the defined URL
driver.get(url)

# Use WebDriverWait to explicitly wait for the presence of at least one movie list item
try:
    WebDriverWait(driver, 30).until( # Increased timeout for robustness
        EC.presence_of_element_located((By.CSS_SELECTOR, 'li.ipc-metadata-list-summary-item'))
    )
    print("Initial IMDb page loaded and at least one movie list item is present.")
except TimeoutException:
    print("Timed out waiting for any movie list item to load. Check CSS selector or page structure.")
    # Print page source to help debug if elements are genuinely missing
    print("Page source:\n", driver.page_source[:1000]) # Print first 1000 characters of page source

# --- End Re-initialize WebDriver and navigate to URL ---


# Define a function to click the "Load More" button
def click_load_more():
    try:
        # Updated XPath for the 'Load More' button
        load_more_button = driver.find_element(By.XPATH, "//button[contains(@class, 'ipc-see-more__button') and contains(span/span, 'more')]")

        # Scroll to the "Load More" button to make sure it's in view
        ActionChains(driver).move_to_element(load_more_button).perform()

        # Click the button
        load_more_button.click()

        # Use a fixed sleep time to allow new content to load
        time.sleep(5) # Increased sleep time for better content loading
        return True
    except (NoSuchElementException, ElementClickInterceptedException) as e:
        print("Load More button not found or not clickable:", e)
        return False
    except Exception as e:
        print("Error clicking 'Load More' button:", e)
        return False

# Now, re-run the scraping loop with the updated click_load_more function
MAX_MOVIES = 5000

# Clear previously collected data before restarting the loop
titles = []
storylines = []

# Initial data extraction before any 'Load More' clicks
print("Starting initial data extraction...")
initial_movie_items = driver.find_elements(By.CSS_SELECTOR, 'li.ipc-metadata-list-summary-item')
for movie_item in initial_movie_items:
    try:
        # Extract Title
        title = movie_item.find_element(By.CSS_SELECTOR, 'h3.ipc-title__text').text

        # Extract Storyline (assign 'N/A' if not found)
        try:
            storyline = movie_item.find_element(By.CSS_SELECTOR, 'div.ipc-html-content-inner-div').text
        except NoSuchElementException:
            storyline = "N/A"



        titles.append(title)
        storylines.append(storyline)

    except Exception as e:
        print(f"Error extracting data for a movie item: {e}")
        continue

print(f"Initially collected {len(titles)} movies.")

# Loop to click 'Load More' and extract data
while len(titles) < MAX_MOVIES:
    print(f"Current movie count: {len(titles)}. Attempting to click 'Load More'...")
    if click_load_more():
        # Get all movie items, including newly loaded ones
        current_movie_items = driver.find_elements(By.CSS_SELECTOR, 'li.ipc-metadata-list-summary-item')

        # Only process items that haven't been processed yet
        for movie_item in current_movie_items[len(titles):]: # Start from the last processed index
            try:
                # Extract Title
                title = movie_item.find_element(By.CSS_SELECTOR, 'h3.ipc-title__text').text

                # Extract Storyline (assign 'N/A' if not found)
                try:
                    storyline = movie_item.find_element(By.CSS_SELECTOR, 'div.ipc-html-content-inner-div').text
                except NoSuchElementException:
                    storyline = "N/A"



                titles.append(title)
                storylines.append(storyline)

                if len(titles) >= MAX_MOVIES:
                    print(f"Reached maximum desired movies ({MAX_MOVIES}). Stopping.")
                    break

            except Exception as e:
                print(f"Error extracting data for a new movie item: {e}")
                continue
    else:
        print("No more 'Load More' button found or clickable. Stopping scraping.")
        break

print(f"Total movies collected: {len(titles)}.")

# Create a DataFrame using the extracted data
df = pd.DataFrame({
    'Title': titles,
    'Storyline': storylines
})

# Save the DataFrame to a CSV file
df.to_csv('movies_2024.csv', index=False)

# Display the first few rows of the DataFrame
print("DataFrame created and saved to 'imdb_movies_2024.csv'. Here are the first 5 rows:")
print(df.head())

# Close the WebDriver
driver.quit()

Initial IMDb page loaded and at least one movie list item is present.
Starting initial data extraction...
Initially collected 50 movies.
Current movie count: 50. Attempting to click 'Load More'...
Current movie count: 100. Attempting to click 'Load More'...
Current movie count: 150. Attempting to click 'Load More'...
Current movie count: 200. Attempting to click 'Load More'...
Current movie count: 250. Attempting to click 'Load More'...
Current movie count: 300. Attempting to click 'Load More'...
Current movie count: 350. Attempting to click 'Load More'...
Current movie count: 400. Attempting to click 'Load More'...
Current movie count: 450. Attempting to click 'Load More'...
Current movie count: 500. Attempting to click 'Load More'...
Current movie count: 550. Attempting to click 'Load More'...
Current movie count: 600. Attempting to click 'Load More'...
Current movie count: 650. Attempting to click 'Load More'...
Current movie count: 700. Attempting to click 'Load More'...
Current mo

## STEP 2: Data Pre-Processing

In [8]:
import pandas as pd
df = pd.read_csv('movies_2024.csv')
df.head()

,Movie Name,Storyline
0,The Substance,A fading celebrity takes a black-market drug: ...
1,The Life of Chuck,"A life-affirming, genre-bending story about th..."
2,Bone Lake,A couple's vacation at a secluded estate is up...
3,Anora,A young stripper from Brooklyn meets and impul...
4,Eden,Based on a factual account of a group of outsi...


In [17]:
df1 = df.copy()
df1.head()

,Movie Name,Storyline
0,The Substance,A fading celebrity takes a black-market drug: ...
1,The Life of Chuck,"A life-affirming, genre-bending story about th..."
2,Bone Lake,A couple's vacation at a secluded estate is up...
3,Anora,A young stripper from Brooklyn meets and impul...
4,Eden,Based on a factual account of a group of outsi...


In [18]:
df1.shape ## The shape of this dataset

(5099, 2)

In [15]:
df1.isna().sum() ## There is no null value

Movie Name    0
Storyline     0
dtype: int64

In [16]:
df1.duplicated().sum() ## There is no duplicate value

0

In [19]:
df1.info() ## It is info of dataset

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5099 entries, 0 to 5098
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Movie Name  5099 non-null   object
 1   Storyline   5099 non-null   object
dtypes: object(2)
memory usage: 79.8+ KB


## STEP 3: NLP(Tokenization -> Remove Stepwords)

In [6]:
import re
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    words = text.split()
    words = [word for word in words if word not in stop_words]
    return " ".join(words)

df1["Storyline"] = df1["Storyline"].apply(clean_text)

#print(df[["Storyline"]].head()
df1.head()

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\kshan\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


,Movie Name,Storyline
0,The Substance,fading celebrity takes blackmarket drug cellre...
1,The Life of Chuck,lifeaffirming genrebending story three chapter...
2,Bone Lake,couples vacation secluded estate upended theyr...
3,Anora,young stripper brooklyn meets impulsively marr...
4,Eden,based factual account group outsiders settle r...


## STEP 4: TF-IDF(Convert Text into Numerical Vector)

In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(df1["Storyline"])

print(tfidf_matrix.shape)

(5099, 18187)


## STEP 5: Cosine-Similarity for Reccomendation

In [8]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(tfidf_matrix)

print(similarity_matrix.shape)

(5099, 5099)


## 1. Predict Movie According to Similar Movie

In [9]:
def recommend(movie_title):
    index = df1[df1["Movie Name"] == movie_title].index[0]
    scores = list(enumerate(similarity_matrix[index]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    top_movies = scores[1:6]

    for i in top_movies:
        print(df1.iloc[i[0]]["Movie Name"])

In [16]:
recommend("Family Padam")

2 Fingers Honey 2
Sunny
Survive
When Fucking Spring Is in the Air
Goreyan Naal Lagdi Zameen Jatt Di


## 2. Predicting Movie According to Storyline

In [11]:
def recommend_from_storyline(user_input):
    cleaned_input = clean_text(user_input)
    input_vector = vectorizer.transform([cleaned_input])
    
    similarity_scores = cosine_similarity(input_vector, tfidf_matrix)
    
    scores = list(enumerate(similarity_scores[0]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)
    
    top_movies = scores[:5]
    
    for i in top_movies:
        print(df1.iloc[i[0]]["Movie Name"])

In [15]:
recommend_from_storyline("Hollywood movie")	

MaXXXine
Family Padam
The Seductress from Hell
Kangaroo Island
Laufey's A Night at the Symphony: Hollywood Bowl
